In [1]:
import random
import ipywidgets as widgets
from IPython.display import display, clear_output
from qiskit import QuantumCircuit, execute, Aer
from qiskit.providers.aer import QasmSimulator
import numpy as np

# Global variables to store the ciphertext and encryption key
global ciphertext_bits, encryption_key
ciphertext_bits = ""
encryption_key = ""

def alice_state_prep(state, basis):
    """
    Prepare Alice's quantum state based on the given state and basis choices.
    :param state: List representing Alice's qubit states (0 or 1).
    :param basis: List representing Alice's basis choices (0 for computational, 1 for Hadamard).
    :return: Quantum circuit representing the prepared state.
    """
    num_qubits = len(state)
    circuit = QuantumCircuit(num_qubits)
    for i in range(len(basis)):
        if state[i] == 1:
            circuit.x(i)  # Apply X gate if state is 1
        if basis[i] == 1:
            circuit.h(i)  # Apply Hadamard gate if basis is 1
    return circuit

def bob_measurement(circuit, basis):
    """
    Apply Bob's measurement operations based on his chosen basis.
    :param circuit: Quantum circuit prepared by Alice.
    :param basis: List representing Bob's basis choices (0 for computational, 1 for Hadamard).
    """
    for i in range(len(basis)):
        if basis[i] == 1:
            circuit.h(i)  # Apply Hadamard gate if basis is 1
    circuit.measure_all()  # Perform measurement on all qubits

def key_creation(circuit, alice_basis, bob_basis):
    """
    Generate an encryption key by comparing Alice's and Bob's basis choices.
    :param circuit: Quantum circuit with measured qubits.
    :param alice_basis: Alice's basis choices.
    :param bob_basis: Bob's basis choices.
    :return: Encryption key as a binary string.
    """
    backend = QasmSimulator()
    result = execute(circuit, backend=backend, shots=1).result()
    counts = result.get_counts()
    key = list(counts.keys())[0] if counts else ""
    encryption_key = ''.join([key[i] for i in range(min(len(alice_basis), len(key))) if alice_basis[i] == bob_basis[i]])
    return encryption_key

def text_to_bits(text):
    """
    Convert a plaintext string into a binary string.
    :param text: String input.
    :return: Binary string representation of the input text.
    """
    return ''.join(format(ord(char), '08b') for char in text)

def encrypt_message(_):
    """
    Encrypt a user-provided text message using quantum key distribution.
    """
    global ciphertext_bits, encryption_key
    plaintext = text_input.value.strip()
    if not plaintext:
        with output:
            clear_output(wait=True)
            print("Please enter a message to encrypt.")
        return
    
    plain_text_bits = text_to_bits(plaintext)  # Convert plaintext to binary
    num_qubits = int(2.0 * len(plain_text_bits))  # Set number of qubits
    print(num_qubits)
    
    # Alice prepares her quantum states
    alice_basis = np.random.randint(2, size=num_qubits)
    alice_state = np.random.randint(2, size=num_qubits)
    cir = alice_state_prep(alice_state, alice_basis)
    
    # Bob chooses his measurement bases and measures the qubits
    bob_basis = np.random.randint(2, size=num_qubits)
    bob_measurement(cir, bob_basis)
    
    # Generate encryption key
    encryption_key = key_creation(cir, alice_basis, bob_basis)
    encryption_key = encryption_key[:len(plain_text_bits)]  # Trim key length to match message
    
    # Encrypt the message using XOR operation
    ciphertext_bits = ''.join(str(int(plain_text_bits[i]) ^ int(encryption_key[i])) for i in range(len(plain_text_bits)))
    
    # Display the encryption results
    clear_output(wait=True)
    display(text_input, encrypt_button, decrypt_button, output)
    with output:
        print("Ciphertext (in binary):", ciphertext_bits)
        print("Key:", encryption_key)

def bits_to_text(bits):
    """
    Convert a binary string back into a plaintext string.
    :param bits: Binary string input.
    :return: Decoded plaintext string.
    """
    chars = [chr(int(bits[i:i+8], 2)) for i in range(0, len(bits), 8)]
    return ''.join(chars)

def decrypt_message(_):
    """
    Decrypt the previously encrypted message using the stored encryption key.
    """
    global ciphertext_bits, encryption_key
    if not ciphertext_bits or not encryption_key:
        with output:
            clear_output(wait=True)
            print("No encrypted message found. Please encrypt a message first.")
        return
    
    # Decrypt message using XOR
    decrypted_bits = ''.join(str(int(ciphertext_bits[i]) ^ int(encryption_key[i])) for i in range(len(ciphertext_bits)))
    decrypted_text = bits_to_text(decrypted_bits)
    
    # Display the decrypted text
    clear_output(wait=True)
    display(text_input, encrypt_button, decrypt_button, output)
    with output:
        print("Decrypted Text:", decrypted_text)

# Create UI elements for user interaction
text_input = widgets.Text(description="Enter Text:", placeholder="Type your message here")
encrypt_button = widgets.Button(description="Encrypt Message")
decrypt_button = widgets.Button(description="Decrypt Message")
encrypt_button.on_click(encrypt_message)
decrypt_button.on_click(decrypt_message)
output = widgets.Output()

# Display UI elements
display(text_input, encrypt_button, decrypt_button, output)


Text(value='', description='Enter Text:', placeholder='Type your message here')

Button(description='Encrypt Message', style=ButtonStyle())

Button(description='Decrypt Message', style=ButtonStyle())

Output()